# Chuẩn bị dữ liệu và chia tập độc lập
Chạy sau eda.ipynb.   
Giữ đủ 12 đặc trưng; không chuẩn hóa, chọn đặc trưng

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / 'data' / 'winequality-red.csv').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Đặt notebook trong dự án ML-report có thư mục data.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
OUT = PROJECT_ROOT / 'output'
OUT.mkdir(exist_ok=True)
from sklearn.model_selection import train_test_split
from src.pipelines import FEATURES, PHYSICO, CLASS_NAMES
df = pd.read_csv(OUT / 'wine_quality_clean.csv')
assert set(df['wine_type'].dropna().unique()) <= {'red', 'white'}
assert df['wine_type'].notna().all()
df['wine_type_white'] = df['wine_type'].map({'red': 0, 'white': 1})
assert set(df['label'].unique()) == {0, 1, 2}
assert set(FEATURES).isdisjoint({'quality', 'quality_group', 'label'})
expected = np.where(df['quality'] <= 5, 0, np.where(df['quality'] == 6, 1, 2))
assert np.array_equal(expected, df['label'].to_numpy())
print('Dữ liệu sạch:', df.shape)


Dữ liệu sạch: (5320, 16)


## 1. Kiểm tra trùng trước chia tập


In [2]:
duplicate_features = df.duplicated(subset=FEATURES, keep=False)
print('Số dòng cùng đặc trưng:', int(duplicate_features.sum()))
if duplicate_features.any():
    display(df.loc[duplicate_features, FEATURES + ['label']])
    raise ValueError('Có trùng đặc trưng: cần kiểm tra và chia nhóm; không tự xóa nhãn.')
X = df[FEATURES].copy()
y = df['label'].astype(int)
assert X.shape[1] == 12
assert not np.isinf(X.to_numpy(dtype=float)).any()
display(y.value_counts().sort_index().rename('count').to_frame())


Số dòng cùng đặc trưng: 0


,count
label,
0,1988
1,2323
2,1009


## 2. Chia train và test
Phân tầng theo label, tỷ lệ 80/20, random_state=42. Chỉ số dòng dữ liệu sạch được lưu để truy vết mẫu sai. Phân tầng theo nhãn không bảo đảm tỷ lệ loại rượu, vì vậy kiểm tra riêng.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
assert X_train.index.intersection(X_test.index).empty
assert len(X_train) + len(X_test) == len(df)
distribution = pd.DataFrame({
    'train_count': y_train.value_counts().sort_index(),
    'test_count': y_test.value_counts().sort_index(),
    'train_percent': y_train.value_counts(normalize=True).sort_index() * 100,
    'test_percent': y_test.value_counts(normalize=True).sort_index() * 100})
display(distribution)
print('Train:', X_train.shape, '| Test:', X_test.shape)
display(pd.DataFrame({'train': df.loc[X_train.index, 'wine_type'].value_counts(normalize=True),
                      'test': df.loc[X_test.index, 'wine_type'].value_counts(normalize=True)}))


,train_count,test_count,train_percent,test_percent
label,,,,
0,1591,397,37.382519,37.312030
1,1858,465,43.656015,43.703008
2,807,202,18.961466,18.984962


Train: (4256, 12) | Test: (1064, 12)


,train,test
wine_type,,
white,0.740132,0.762218
red,0.259868,0.237782


## 3. Lưu dữ liệu chưa chuẩn hóa và cấu hình
Các notebook huấn luyện sử dụng Pipeline để học trung vị và chuẩn hóa trong từng fold.

In [4]:
for name, data in [('X_train', X_train), ('X_test', X_test),
                   ('y_train', y_train), ('y_test', y_test)]:
    data.to_csv(OUT / f'{name}.csv', index=False)
for split, frame in [('train', X_train), ('test', X_test)]:
    meta = df.loc[frame.index, ['wine_type', 'quality']].copy()
    meta.insert(0, 'clean_row_id', frame.index)
    meta.to_csv(OUT / f'{split}_metadata.csv', index=False)
distribution.to_csv(OUT / 'split_distribution.csv')
config = dict(random_state=42, test_size=0.2, n_splits=5,
              features=FEATURES, target='label', class_names=CLASS_NAMES,
              main_metric='f1_macro', feature_selection=False,
              preprocessing='inside_pipeline', cv_std_ddof=1)
(OUT / 'config.json').write_text(json.dumps(config, indent=2, ensure_ascii=False), encoding='utf-8')
print('Đã lưu đủ 12 đặc trưng, chưa chuẩn hóa.')


Đã lưu đủ 12 đặc trưng, chưa chuẩn hóa.
